# 01 - OCI Vision AI Quickstart

A five-minute grand tour of OCI Vision AI. We will analyze a single image
with **all available features** (classification + object detection), display
classification confidence bars, a detection results table, and an annotated
overlay image -- all without needing real OCI credentials.

## Setup

In [ ]:
# Install the package (skip if already installed)
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient
from oci_vision.core.renderer import render_overlay
from PIL import Image

# Demo mode -- no OCI credentials required
client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

In [ ]:
# Analyze a gallery image with all features
report = client.analyze("dog_closeup.jpg", features="all")

print(f"Image:    {report.image_path}")
print(f"Features: {report.available_features}")
print(f"Elapsed:  {report.elapsed_seconds:.3f}s")

## Explore the results

### Classification labels

In [ ]:
# Top 10 classification labels
for i, label in enumerate(report.classification.labels[:10], 1):
    bar = '#' * int(label.confidence * 40)
    print(f"{i:2d}. {label.name:20s} {label.confidence_pct:6.2f}% {bar}")

### Classification bar chart

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

top = report.classification.labels[:10]
names = [l.name for l in reversed(top)]
scores = [l.confidence_pct for l in reversed(top)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(names, scores, color='#0077B6')
ax.set_xlabel('Confidence (%)')
ax.set_title('Top 10 Classification Labels')
ax.set_xlim(0, 105)
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('quickstart_classification.png', dpi=100, bbox_inches='tight')
plt.show()

### Detected objects

In [ ]:
# Detection results as a table
print(f"{'#':>3s}  {'Object':15s}  {'Confidence':>10s}  {'Position'}")
print('-' * 55)
for i, obj in enumerate(report.detection.objects, 1):
    pos = obj.bounding_polygon.human_position(1024, 768)
    print(f"{i:3d}  {obj.name:15s}  {obj.confidence_pct:9.2f}%  {pos}")

## Visualize

In [ ]:
# Load the gallery image and render overlays
from oci_vision.gallery import get_gallery_path

img_path = get_gallery_path() / "images" / "dog_closeup.jpg"
img = Image.open(img_path)
print(f"Original image: {img.size[0]}x{img.size[1]}")

overlay = render_overlay(img, report)
overlay.save('quickstart_overlay.png')
print("Overlay rendered with classification tags + detection boxes")
overlay

## Under the hood

In [ ]:
import json

# Raw JSON representation of the full report
raw = report.model_dump()
print(json.dumps(raw, indent=2, default=str)[:3000])
print('...')  # truncated for readability

## Try it yourself

1. **Switch to live mode**: Replace `VisionClient(demo=True)` with
   `VisionClient()` and analyze your own images (requires OCI credentials).
2. **Filter by confidence**: Try `report.classification.above_threshold(0.95)`
   to keep only high-confidence labels.
3. **Analyze multiple images**: Loop over several files and compare results.
4. **Individual features**: Use `client.classify(img)` or `client.detect_objects(img)`
   instead of `client.analyze()` for single-feature calls.